# 📊 Workforce Attrition Patterns — Exploratory Data Analysis

**Palo Alto Networks — HR Analytics**

This notebook performs a comprehensive EDA on the employee attrition dataset, exploring patterns across demographics, tenure, workload, and satisfaction dimensions.

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import sys, os, warnings
warnings.filterwarnings('ignore')

# Style
sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.1)
plt.rcParams['figure.figsize'] = (10, 5)

# Load project modules
sys.path.insert(0, os.path.abspath('..'))
from src.cleaning import prepare_data
from src.kpis import compute_all_kpis

In [ ]:
# Load and prepare data
df = prepare_data('../data/attrition.csv')
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
# Helper: compute attrition rate by group with sample size
def attrition_by(data, col, sort_rate=True):
    grp = data.groupby(col).agg(
        Employees=('AttritionFlag', 'count'),
        Exited=('AttritionFlag', 'sum')
    ).reset_index()
    grp['Rate(%)'] = (grp['Exited'] / grp['Employees'] * 100).round(1)
    grp['LowSample'] = grp['Employees'] < 30
    if sort_rate:
        grp = grp.sort_values('Rate(%)', ascending=False)
    return grp.reset_index(drop=True)

def bar_with_n(data, x, y='Rate(%)', title='', palette='YlOrRd'):
    """Bar chart with rate and sample size annotation."""
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = sns.barplot(data=data, x=x, y=y, palette=palette, ax=ax, hue=x, dodge=False, legend=False)
    for i, row in data.iterrows():
        label = f"{row[y]}%\n(n={row['Employees']})"
        if row.get('LowSample', False):
            label += ' \u26a0'
        ax.text(i, row[y] + 0.8, label, ha='center', fontsize=9, fontweight='bold')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_ylabel('Attrition Rate (%)')
    plt.tight_layout()
    plt.show()

---
## 1. Overall Attrition Rate

We first examine the headline attrition rate and the split between retained and exited employees.

In [ ]:
kpis = compute_all_kpis(df)
print(f"Overall Attrition Rate: {kpis['overall_rate']}%")
print(f"Total Employees: {kpis['total_employees']}")
print(f"Total Exited: {kpis['total_exited']}")
print(f"Early-Tenure Attrition: {kpis['early_tenure_rate']}%")
print(f"Workload Attrition Index: {kpis['workload_index']}x")

In [ ]:
# Donut chart: Retained vs Exited
counts = df['Attrition'].value_counts()
fig, ax = plt.subplots(figsize=(6, 6))
colors = ['#4facfe', '#f5576c']
wedges, texts, autotexts = ax.pie(
    counts, labels=counts.index, autopct='%1.1f%%',
    colors=colors, startangle=90, pctdistance=0.8,
    wedgeprops=dict(width=0.4)
)
ax.set_title('Retained vs Exited Employees', fontsize=14, fontweight='bold')
centre_circle = plt.Circle((0, 0), 0.55, fc='white')
ax.add_artist(centre_circle)
ax.text(0, 0, f'{kpis["total_employees"]}\nTotal', ha='center', va='center', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Insight:** The overall attrition rate stands at ~26.5%, meaning roughly 1 in 4 employees have left the organization. This is a significant rate that warrants deep investigation into which segments are most affected.

---
## 2. Attrition by Department and Job Role

Which departments and roles are losing the most talent?

In [ ]:
# Department attrition
dept_data = attrition_by(df, 'Department')
bar_with_n(dept_data, 'Department', title='Attrition Rate by Department')
dept_data

**Insight:** Human Resources has the highest attrition rate at ~36.5%, though it has the smallest headcount (n=63). Sales follows at ~30.8%. R&D, the largest department, has the lowest rate at ~24.1%. The HR department's small size means even a few departures create a large percentage impact.

In [ ]:
# Job Role attrition
role_data = attrition_by(df, 'JobRole')
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=role_data, x='Rate(%)', y='JobRole', palette='YlOrRd_r', ax=ax, hue='JobRole', dodge=False, legend=False)
for i, row in role_data.iterrows():
    label = f"{row['Rate(%)']}% (n={row['Employees']})"
    if row['LowSample']:
        label += ' \u26a0'
    ax.text(row['Rate(%)'] + 0.3, i, label, va='center', fontsize=9)
ax.axvline(x=kpis['overall_rate'], color='red', linestyle='--', label=f"Overall Avg: {kpis['overall_rate']}%")
ax.legend()
ax.set_title('Attrition Rate by Job Role (dashed line = overall average)', fontsize=14, fontweight='bold')
ax.set_xlabel('Attrition Rate (%)')
plt.tight_layout()
plt.show()
role_data

**Insight:** HR roles and Sales Representatives show the highest attrition (37.3% and 31.7% respectively). Roles above the overall average line are flagged as high-risk. Research Directors and Healthcare Representatives have the lowest rates (~21-22%).

---
## 3. Department × Job Role Heatmap

In [ ]:
# Department x JobRole heatmap
pivot = df.pivot_table(values='AttritionFlag', index='Department', columns='JobRole', aggfunc='mean') * 100
fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot.round(1), annot=True, fmt='.1f', cmap='YlOrRd', linewidths=0.5,
            cbar_kws={'label': 'Attrition Rate (%)'}, ax=ax)
ax.set_title('Attrition Rate (%) \u2014 Department \u00d7 Job Role', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Insight:** The heatmap reveals specific hotspots where department and role intersect. NaN cells indicate role-department combinations that don't exist in the organization. The darkest cells represent the highest-risk intersections that need targeted retention strategies.

---
## 4. Demographics: Age, Gender, Marital Status, Education

In [ ]:
# Age Group
age_data = attrition_by(df, 'AgeGroup', sort_rate=False)
bar_with_n(age_data, 'AgeGroup', title='Attrition Rate by Age Group', palette='Reds')
age_data

**Insight:** The youngest cohort (18-25) shows elevated attrition at ~34.3%, consistent with early-career mobility. The 56+ group shows the highest rate (~42.9%), but with only n=21 this should be interpreted cautiously (⚠ low sample). The 46-55 band has the lowest rate at ~22.8%, suggesting mid-to-late career stability.

In [ ]:
# Gender
gender_data = attrition_by(df, 'Gender')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gender bar
sns.barplot(data=gender_data, x='Gender', y='Rate(%)', palette=['#4facfe', '#f093fb'], ax=axes[0], hue='Gender', dodge=False, legend=False)
for i, row in gender_data.iterrows():
    axes[0].text(i, row['Rate(%)'] + 0.5, f"{row['Rate(%)']}% (n={row['Employees']})", ha='center', fontweight='bold')
axes[0].set_title('Attrition Rate by Gender', fontweight='bold')
axes[0].set_ylabel('Attrition Rate (%)')

# Marital Status
ms_data = attrition_by(df, 'MaritalStatus')
sns.barplot(data=ms_data, x='MaritalStatus', y='Rate(%)', palette='YlOrRd', ax=axes[1], hue='MaritalStatus', dodge=False, legend=False)
for i, row in ms_data.iterrows():
    axes[1].text(i, row['Rate(%)'] + 0.5, f"{row['Rate(%)']}% (n={row['Employees']})", ha='center', fontweight='bold')
axes[1].set_title('Attrition Rate by Marital Status', fontweight='bold')
axes[1].set_ylabel('Attrition Rate (%)')

plt.tight_layout()
plt.show()

**Insight:** Males have a slightly higher attrition rate (~27.8%) compared to females (~24.7%). However, marital status shows a much stronger pattern: single employees leave at 33.1% — nearly double the rate of divorced employees (19.5%). This likely reflects career-stage and mobility factors.

In [ ]:
# Education Level and Education Field
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

edu_data = attrition_by(df, 'EducationLabel')
sns.barplot(data=edu_data, x='EducationLabel', y='Rate(%)', palette='Blues_d', ax=axes[0], hue='EducationLabel', dodge=False, legend=False)
for i, row in edu_data.iterrows():
    axes[0].text(i, row['Rate(%)'] + 0.5, f"{row['Rate(%)']}%\n(n={row['Employees']})", ha='center', fontsize=8)
axes[0].set_title('Attrition Rate by Education Level', fontweight='bold')
axes[0].set_ylabel('Attrition Rate (%)')
axes[0].tick_params(axis='x', rotation=30)

ef_data = attrition_by(df, 'EducationField')
sns.barplot(data=ef_data, x='EducationField', y='Rate(%)', palette='viridis', ax=axes[1], hue='EducationField', dodge=False, legend=False)
for i, row in ef_data.iterrows():
    axes[1].text(i, row['Rate(%)'] + 0.5, f"{row['Rate(%)']}%\n(n={row['Employees']})", ha='center', fontsize=8)
axes[1].set_title('Attrition Rate by Education Field', fontweight='bold')
axes[1].set_ylabel('Attrition Rate (%)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

**Insight:** Education level differences are relatively modest. Bachelor's degree holders show the highest attrition (~28.9%) while Below College is lowest (~22.1%). Education field shows more variation — worth monitoring which fields correspond to harder-to-fill positions.

---
## 5. Tenure Analysis: Years at Company, Career Stage, Promotion Stagnation

In [ ]:
# Tenure Bucket & Career Stage
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

tb_data = attrition_by(df, 'TenureBucket', sort_rate=False)
sns.barplot(data=tb_data, x='TenureBucket', y='Rate(%)', palette='Reds', ax=axes[0], hue='TenureBucket', dodge=False, legend=False)
for i, row in tb_data.iterrows():
    axes[0].text(i, row['Rate(%)'] + 0.5, f"{row['Rate(%)']}%\n(n={row['Employees']})", ha='center', fontweight='bold', fontsize=9)
axes[0].set_title('Attrition Rate by Tenure Bucket', fontweight='bold')
axes[0].set_ylabel('Attrition Rate (%)')

cs_data = attrition_by(df, 'CareerStage', sort_rate=False)
order = ['Early', 'Mid', 'Senior']
cs_data['CareerStage'] = pd.Categorical(cs_data['CareerStage'], categories=order, ordered=True)
cs_data = cs_data.sort_values('CareerStage')
sns.barplot(data=cs_data, x='CareerStage', y='Rate(%)', palette='YlOrRd', ax=axes[1], hue='CareerStage', dodge=False, legend=False)
for i, row in cs_data.iterrows():
    axes[1].text(list(range(len(cs_data))).pop(0) if i == cs_data.index[0] else (1 if i == cs_data.index[1] else 2),
                 row['Rate(%)'] + 0.5,
                 f"{row['Rate(%)']}%\n(n={row['Employees']})", ha='center', fontweight='bold', fontsize=9)
axes[1].set_title('Attrition Rate by Career Stage', fontweight='bold')
axes[1].set_ylabel('Attrition Rate (%)')

plt.tight_layout()
plt.show()

**Insight:** There is a very clear inverse relationship between tenure and attrition. Employees in their first 0-2 years have a 33.3% attrition rate — more than double the 10+ year cohort (16.7%). This early-tenure attrition represents the biggest single risk area, suggesting onboarding and early engagement programs are critical.

In [ ]:
# Promotion Stagnation
ps_data = attrition_by(df, 'PromotionStagnation', sort_rate=False)
bar_with_n(ps_data, 'PromotionStagnation', title='Attrition Rate by Years Since Last Promotion', palette='Oranges')
ps_data

**Insight:** Interestingly, employees promoted most recently (0-1 years ago) show the highest attrition (~27.2%). This could indicate that recent promotions sometimes trigger market re-evaluation, or that recent promotees with updated resumes are more attractive to recruiters. The 5+ years since promotion group actually has lower attrition (~18.6%), though this group is small (n=43).

---
## 6. Workload: OverTime, Business Travel, Distance, Combined Heatmap

In [ ]:
# OverTime and Business Travel
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ot_data = attrition_by(df, 'OverTime')
colors_ot = ['#f5576c' if v == 'Yes' else '#4facfe' for v in ot_data['OverTime']]
sns.barplot(data=ot_data, x='OverTime', y='Rate(%)', palette=colors_ot, ax=axes[0], hue='OverTime', dodge=False, legend=False)
for i, row in ot_data.iterrows():
    axes[0].text(i, row['Rate(%)'] + 0.8, f"{row['Rate(%)']}%\n(n={row['Employees']})", ha='center', fontweight='bold')
axes[0].set_title('Attrition Rate: OverTime Impact', fontweight='bold')
axes[0].set_ylabel('Attrition Rate (%)')

bt_data = attrition_by(df, 'BusinessTravel')
sns.barplot(data=bt_data, x='BusinessTravel', y='Rate(%)', palette='YlOrRd', ax=axes[1], hue='BusinessTravel', dodge=False, legend=False)
for i, row in bt_data.iterrows():
    axes[1].text(i, row['Rate(%)'] + 0.8, f"{row['Rate(%)']}%\n(n={row['Employees']})", ha='center', fontweight='bold')
axes[1].set_title('Attrition Rate by Business Travel', fontweight='bold')
axes[1].set_ylabel('Attrition Rate (%)')

plt.tight_layout()
plt.show()

**Insight:** OverTime is the single strongest predictor of attrition in this dataset. Employees working overtime leave at 46.3% — nearly 2.5x the rate of those who don't (18.9%). Frequent travelers also show elevated risk at 32.0%, compared to 22.6% for non-travelers. These are actionable factors that the organization can directly influence through workload management.

In [ ]:
# Distance from Home
db_data = attrition_by(df, 'DistanceBand', sort_rate=False)
bar_with_n(db_data, 'DistanceBand', title='Attrition Rate by Distance from Home', palette='Purples')
db_data

**Insight:** Employees living far from the office (16+ km) have a moderately higher attrition rate (~31.7%) compared to those nearby (~25.5-25.7%). Distance is a contributing factor but not as strong as overtime or travel frequency.

In [ ]:
# OverTime x BusinessTravel Combined Heatmap
pivot_ot_bt = df.pivot_table(values='AttritionFlag', index='OverTime', columns='BusinessTravel', aggfunc=['mean', 'count'])
rates = (pivot_ot_bt['mean'] * 100).round(1)
counts = pivot_ot_bt['count']

# Build annotation with rate and count
annot = rates.copy().astype(str)
for col in annot.columns:
    for idx in annot.index:
        n = int(counts.loc[idx, col])
        annot.loc[idx, col] = f"{rates.loc[idx, col]}%\n(n={n})"

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(rates, annot=annot, fmt='', cmap='YlOrRd', linewidths=1,
            cbar_kws={'label': 'Attrition Rate (%)'}, ax=ax)
ax.set_title('Attrition Rate (%) \u2014 OverTime \u00d7 Business Travel', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Insight:** The combined heatmap reveals a critical hotspot: employees who work overtime AND travel frequently have a **51.7% attrition rate** (n=87). This is nearly triple the rate of non-overtime, non-travel employees (16.2%). The Workload Attrition Index (WAI) of 1.95x confirms that high-workload employees leave at almost double the overall rate. This is the single most actionable finding in the analysis.

---
## 7. Satisfaction Scores & Income Analysis

In [ ]:
# Satisfaction scores
sat_cols = ['EnvironmentSatisfaction', 'JobSatisfaction', 'JobInvolvement', 'WorkLifeBalance']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for i, col in enumerate(sat_cols):
    sat_data = attrition_by(df, col, sort_rate=False)
    sns.barplot(data=sat_data, x=col, y='Rate(%)', palette='RdYlGn', ax=axes[i], hue=col, dodge=False, legend=False)
    for j, row in sat_data.iterrows():
        axes[i].text(j, row['Rate(%)'] + 0.5, f"{row['Rate(%)']}%\n(n={row['Employees']})", ha='center', fontsize=8)
    axes[i].set_title(f'Attrition by {col}', fontweight='bold')
    axes[i].set_ylabel('Rate (%)')
    axes[i].set_xlabel(f'{col} (1=Low, 4=High)')

plt.suptitle('Attrition Rate by Satisfaction Scores', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Insight:** Across all satisfaction dimensions, employees rating their satisfaction as 1 (lowest) consistently show the highest attrition rates:
- Environment Satisfaction = 1: ~36.6%
- Job Satisfaction = 1: ~35.7%
- Job Involvement = 1: ~34.4%
- Work-Life Balance = 1: ~34.7%

This validates that employee engagement surveys can serve as early warning signals for attrition risk.

In [ ]:
# Monthly Income vs Attrition
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x='Attrition', y='MonthlyIncome', palette=['#4facfe', '#f5576c'], ax=ax, hue='Attrition', dodge=False, legend=False)
ax.set_title('Monthly Income Distribution by Attrition Status', fontweight='bold')
ax.set_ylabel('Monthly Income')

# Add median annotations
medians = df.groupby('Attrition')['MonthlyIncome'].median()
for i, (label, med) in enumerate(medians.items()):
    ax.text(i, med + 200, f'Median: ${med:,.0f}', ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

**Insight:** Employees who leave tend to have lower monthly incomes than those who stay. The median income gap suggests that compensation competitiveness plays a role in retention, particularly for lower-level positions.

In [ ]:
# Correlation heatmap of numeric features with AttritionFlag
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_with_attrition = df[numeric_cols].corr()['AttritionFlag'].drop('AttritionFlag').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#f5576c' if v > 0 else '#4facfe' for v in corr_with_attrition]
corr_with_attrition.plot(kind='barh', color=colors, ax=ax)
ax.set_title('Correlation of Numeric Features with Attrition', fontweight='bold', fontsize=14)
ax.set_xlabel('Pearson Correlation Coefficient')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

**Insight:** The correlation analysis confirms our EDA findings. Features most positively correlated with attrition include factors related to workload and early career, while tenure-related features (YearsAtCompany, YearsInCurrentRole) are negatively correlated, confirming that longer-tenured employees are more stable.

---
## Summary of Key Findings

| Risk Factor | Attrition Rate | vs. Overall (26.5%) |
|---|---|---|
| OverTime = Yes | 46.3% | +19.8pp |
| OT + Frequent Travel | 51.7% | +25.2pp |
| Early Tenure (0-2 yrs) | 33.3% | +6.8pp |
| Single employees | 33.1% | +6.6pp |
| Age 18-25 | 34.3% | +7.8pp |
| HR Department | 36.5% | +10.0pp |
| Low Satisfaction (any=1) | 34-37% | +8-11pp |
| Far Distance (16+ km) | 31.7% | +5.2pp |

**Top 3 actionable recommendations:**
1. **Reduce overtime burden** — strongest single predictor; 46.3% vs 18.9%.
2. **Strengthen first-2-year experience** — onboarding, mentorship, check-ins.
3. **Monitor high-workload combinations** — OT + frequent travel hits 51.7%.